# Earnalism stills job `selfish-giant-stills-film-01-house`

**Prepared, not run.** Input zip SHA256 `8ea0cfdb3c8bd6d9e7c3a244138ac99a64594f2c2ebfe72eca278054d6ebfe22`. Set `RUN = True` in the first cell, then run all. With the R2 exchange (`exchange.json` in `/content/drive/MyDrive/earnalism-audiovisual/the-selfish-giant/selfish-giant-stills-film-01-house/`) the notebook downloads `inputs.zip` itself and uploads the outputs when it finishes; without it, copy `inputs.zip` into that Drive folder first. Outputs and `manifest.json` (sha256 of every output) land in the same folder.

**Hard stop:** 4024 s (the reserved forecast) from the clock cell. No item starts after it and the runtime is unassigned at it; a stopped run's manifest says INCOMPLETE and cannot be imported.


In [ ]:
RUN = False

In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import pathlib
JOB_ID = 'selfish-giant-stills-film-01-house'
TITLE = 'the-selfish-giant'
KIND = 'stills'
INPUT_ZIP_SHA256 = '8ea0cfdb3c8bd6d9e7c3a244138ac99a64594f2c2ebfe72eca278054d6ebfe22'
MODEL_REPO = 'black-forest-labs/FLUX.1-schnell'
MODEL_REVISION = 'resolved-at-run-time'
STEPS = 4
DRIVE_JOB_DIR = '/content/drive/MyDrive/earnalism-audiovisual/the-selfish-giant/selfish-giant-stills-film-01-house'
# Hard wall-clock stop = the forecast seconds reserved on the ledger (R25)
DEADLINE_SECONDS = 4024
MANIFEST_RESERVE_S = 60
PACKAGES = ('diffusers', 'transformers', 'accelerate', 'sentencepiece', 'protobuf', 'bitsandbytes', 'torch')
EXCHANGE_SCHEMA = 'earnalism.colab-exchange.v1'


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import os
_token, _secret_problem = None, None
try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN") or None
except Exception as _e:  # no secret, notebook access not granted, or the prompt timed out
    _secret_problem = type(_e).__name__
    _token = os.environ.get("HF_TOKEN") or None
if _token:
    os.environ["HF_TOKEN"] = _token
from huggingface_hub import HfApi
try:
    HfApi().auth_check(MODEL_REPO, token=_token)
except Exception as _e:
    _fix = ("accept the model's terms at https://huggingface.co/" + MODEL_REPO + " with your Hugging "
            "Face account, put a Read token from https://huggingface.co/settings/tokens in the Colab "
            "secret HF_TOKEN (key icon in the left sidebar, Notebook access on), then Runtime > "
            "Restart session and Run all")
    if _secret_problem:
        _fix += " (the HF_TOKEN secret could not be read: %s; grant access if Colab asks)" % _secret_problem
    raise RuntimeError("No access to " + MODEL_REPO + " (" + type(_e).__name__ + "): " + _fix) from None
print("model access OK:", MODEL_REPO, "(token from HF_TOKEN)" if _token else "(public)")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import threading, time
if "JOB_STARTED" not in globals():
    JOB_STARTED = time.time()
    from google.colab import runtime
    STOP_GUARD = threading.Timer(DEADLINE_SECONDS, runtime.unassign)
    STOP_GUARD.daemon = True
    STOP_GUARD.start()
INCOMPLETE = False


def may_start(last_item_s):
    """Between items: start the next one only if it (and the manifest) fits before the hard stop."""
    global INCOMPLETE
    left = DEADLINE_SECONDS - (time.time() - JOB_STARTED)
    if left < last_item_s + MANIFEST_RESERVE_S:
        INCOMPLETE = True
        print("HARD STOP: %.0f s left of the %d s forecast; no further item starts" % (left, DEADLINE_SECONDS))
        return False
    return True


print("hard stop armed:", DEADLINE_SECONDS, "s from now")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, pathlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = pathlib.Path(DRIVE_JOB_DIR)
INPUT_ZIP = OUT_DIR / "inputs.zip"
# R2 exchange (references/colab-exchange.md): exchange.json in this Drive folder carries two
# short-lived single-object URLs (GET inputs.zip, PUT outputs.tar). They are capabilities: never
# print them, never copy them elsewhere.
EXCHANGE = None
if (OUT_DIR / "exchange.json").is_file():
    import time, urllib.request
    from datetime import datetime
    EXCHANGE = json.loads((OUT_DIR / "exchange.json").read_text())
    if EXCHANGE.get("schema") != EXCHANGE_SCHEMA or EXCHANGE.get("job_id") != JOB_ID:
        raise RuntimeError("exchange.json belongs to a different job; ask Claude for this job's exchange.json")
    _left = datetime.fromisoformat(EXCHANGE["expires_at"].replace("Z", "+00:00")).timestamp() - time.time()
    if _left < DEADLINE_SECONDS + MANIFEST_RESERVE_S + 300:
        raise RuntimeError("the exchange URLs expire before this job's hard stop (%.0f s left); "
                           "ask Claude to re-sign them before running" % _left)
    if not INPUT_ZIP.is_file():
        with urllib.request.urlopen(EXCHANGE["inputs"]["url"], timeout=600) as _r:
            if not 200 <= _r.status < 300:
                raise RuntimeError("inputs.zip download failed: HTTP %d" % _r.status)
            _data = _r.read()
        _sha = hashlib.sha256(_data).hexdigest()
        if _sha != INPUT_ZIP_SHA256 or _sha != EXCHANGE["inputs"]["sha256"]:
            raise RuntimeError("downloaded inputs.zip sha256 mismatch: " + _sha + " (not kept)")
        INPUT_ZIP.write_bytes(_data)
        print("inputs.zip downloaded from the exchange:", len(_data), "bytes")
if not INPUT_ZIP.is_file():
    raise RuntimeError("Copy inputs.zip into " + str(OUT_DIR) + " first")
actual = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
if actual != INPUT_ZIP_SHA256:
    raise RuntimeError("inputs.zip sha256 mismatch: " + actual)
with zipfile.ZipFile(INPUT_ZIP) as z:
    JOB = json.loads(z.read("job.json"))
if JOB["job_id"] != JOB_ID:
    raise RuntimeError("job.json belongs to a different job")
print("inputs verified for", JOB_ID)


In [ ]:
# Stills: FLUX.1-schnell, 4 steps. The exact revision is resolved now and recorded.
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import os, gc, subprocess, sys
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers",
                "accelerate", "sentencepiece", "protobuf", "bitsandbytes"], check=True)
import torch
from diffusers import BitsAndBytesConfig, FluxPipeline, FluxTransformer2DModel
from huggingface_hub import HfApi

revision = HfApi().model_info(MODEL_REPO).sha
# Two phases so the text encoders (~10 GB) and the transformer never share the GPU.
# The transformer is ~24 GB in bf16, more than an L4 holds (22 GB): below 30 GB of VRAM
# it loads 8-bit (~12 GB, near-lossless); an A100/H100 keeps bf16.
gpu = torch.cuda.get_device_properties(0)
QUANT = "bf16" if gpu.total_memory >= 30 * 2**30 else "int8"
print("GPU", gpu.name, "%.1f GiB;" % (gpu.total_memory / 2**30), "transformer", QUANT)
T5_MAX = 256  # FLUX.1-schnell's T5 limit


def look_first(prompt):
    """CLIP reads only 77 tokens and T5 256: put the house style first so it is never cut."""
    k = prompt.lower().find("earnalism house style")
    if k <= 0:
        return prompt
    return prompt[k:].rstrip() + " " + prompt[:k].rstrip(" ,;")


# Phase 1: encode every prompt with only the text encoders on the GPU.
enc = FluxPipeline.from_pretrained(MODEL_REPO, revision=revision, transformer=None, vae=None,
                                   torch_dtype=torch.bfloat16).to("cuda")
EMBEDS, t5_cut = {}, []
with torch.no_grad():
    for item in JOB["items"]:
        text = look_first(item["prompt"])
        if len(enc.tokenizer_2(text).input_ids) > T5_MAX:
            t5_cut.append(item["key"])
        pe, pooled, _ = enc.encode_prompt(prompt=text, prompt_2=text, device="cuda",
                                          max_sequence_length=T5_MAX)
        EMBEDS[item["key"]] = (pe.cpu(), pooled.cpu())
print("encoded", len(EMBEDS), "prompts; T5 tail cut on", len(t5_cut), t5_cut)
del enc
gc.collect()
torch.cuda.empty_cache()

# Phase 2: the transformer (and the small VAE) alone on the GPU.
kw = {"revision": revision, "torch_dtype": torch.bfloat16}
if QUANT == "int8":
    kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
transformer = FluxTransformer2DModel.from_pretrained(MODEL_REPO, subfolder="transformer", **kw)
if QUANT == "bf16":
    transformer.to("cuda")
assert next(transformer.parameters()).device.type == "cuda", "transformer is not on the GPU"
pipe = FluxPipeline.from_pretrained(MODEL_REPO, revision=revision, transformer=transformer,
                                    text_encoder=None, text_encoder_2=None, tokenizer=None,
                                    tokenizer_2=None, torch_dtype=torch.bfloat16)
pipe.vae.to("cuda")
print("GPU memory after load: %.1f GiB" % (torch.cuda.memory_allocated() / 2**30))
MODEL_INFO = {"repo": MODEL_REPO, "revision": revision, "steps": STEPS, "gpu": gpu.name,
              "transformer_precision": QUANT, "prompt_order": "house style first",
              "t5_truncated": t5_cut}
EXPECTED = [item["key"] + ".png" for item in JOB["items"]]
last = 0.0  # seconds the previous still took; the next starts only if it fits (R25)
for item in JOB["items"]:
    if not may_start(last):
        break
    started = time.time()
    gen = torch.Generator("cpu").manual_seed(item["seed"])
    pe, pooled = EMBEDS[item["key"]]
    with torch.no_grad():
        image = pipe(prompt_embeds=pe.to("cuda"), pooled_prompt_embeds=pooled.to("cuda"),
                     num_inference_steps=STEPS, guidance_scale=0.0, max_sequence_length=T5_MAX,
                     width=item["width"], height=item["height"], generator=gen).images[0]
    image.save(OUT_DIR / (item["key"] + ".png"))
    last = time.time() - started
    print("wrote", item["key"], "in %.1f s" % last)


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, time
import importlib.metadata as _md
outputs = {}
for p in sorted(OUT_DIR.iterdir()):
    if (p.is_file() and p.name not in ("manifest.json", "inputs.zip", "exchange.json")
            and p.suffix != ".ipynb"):  # Colab keeps the job notebook in this folder
        outputs[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
if not outputs:
    raise RuntimeError("no outputs were produced")
packages = {}  # R26: the resolved versions (and VCS commits) that actually ran
for _name in PACKAGES:
    try:
        _dist = _md.distribution(_name)
        _url = json.loads(_dist.read_text("direct_url.json") or "{}")
        _commit = _url.get("vcs_info", {}).get("commit_id")
        packages[_name] = _dist.version + (" @ " + _commit if _commit else "")
    except _md.PackageNotFoundError:
        packages[_name] = None
complete = not INCOMPLETE and all(name in outputs for name in EXPECTED)
manifest = {"schema": "earnalism.audiovisual-outputs.v1", "job_id": JOB_ID, "title": TITLE,
            "kind": KIND, "input_zip_sha256": INPUT_ZIP_SHA256, "model": MODEL_INFO,
            "status": "COMPLETE" if complete else "INCOMPLETE",
            "deadline_seconds": DEADLINE_SECONDS,
            "elapsed_seconds": round(time.time() - JOB_STARTED, 1),
            "packages": packages, "outputs": outputs}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print("manifest.json written:", manifest["status"], len(outputs), "files")
if not complete:
    print("INCOMPLETE: import refuses this folder; close the job with the CU Colab shows")
if globals().get("EXCHANGE") is not None:  # R2 exchange: one uncompressed tar, PUT once
    import os, tarfile, tempfile, urllib.error, urllib.request
    _fd, _tar = tempfile.mkstemp(suffix=".tar")  # local disk, never the Drive job folder
    os.close(_fd)
    with tarfile.open(_tar, "w", format=tarfile.PAX_FORMAT) as _t:
        for _name in ["manifest.json"] + sorted(outputs):
            _t.add(str(OUT_DIR / _name), arcname=_name, recursive=False)
    _size = os.path.getsize(_tar)
    _headers = dict(EXCHANGE["outputs"]["headers"], **{"Content-Length": str(_size)})
    with open(_tar, "rb") as _body:
        _req = urllib.request.Request(EXCHANGE["outputs"]["url"], data=_body, method="PUT",
                                      headers=_headers)
        try:
            with urllib.request.urlopen(_req, timeout=1800) as _r:
                _status = _r.status
        except urllib.error.HTTPError as _e:
            _status = _e.code
    os.unlink(_tar)
    print("outputs.tar upload: HTTP", _status, "(%d bytes)" % _size)
    if not 200 <= _status < 300:
        raise RuntimeError("outputs.tar upload failed: HTTP %d; the outputs stay in Drive, tell Claude"
                           % _status)
    print("outputs uploaded to the exchange; tell Claude the CU Colab shows")
